In [ ]:
import pandas as pd
import yfinance as yf
import time
import random
import csv
import logging
from datetime import datetime

In [ ]:
logging.basicConfig(filename = './Logs/resultLogs.log', level = logging.INFO, format = "%(levelname)s - %(message)s", filemode='a')

### Getting the data

In [ ]:
tickers = pd.read_csv('../Data/All_Stocks.csv')
to_clean = pd.DataFrame(tickers)
OUTPUT_PATH = '../Data/EPS_Bigger_Than_PE.csv'
batch_size = 50

In [ ]:
#tickersList = tickers['Symbol'].dropna().unique().tolist()
tickersList = tickers['Symbol'].unique().tolist()

### 'Cleaning' Data

In [ ]:
pattern = r'\^'
rows_to_drop = to_clean[to_clean['Symbol'].str.contains(pattern, na = False)].index
cleaned = to_clean.drop(rows_to_drop)
cleaned['Symbol'] = cleaned['Symbol'].str.replace(r'[\\/]', '-', regex = True)
cleaned['Symbol'] = cleaned['Symbol'].str.strip()
cleaned = cleaned['Symbol'].tolist()

In [ ]:
results = []

counter = 1

In [ ]:
print(len(tickersList))
print(len(cleaned))
cleaned = cleaned

In [ ]:
def check_the_stock (batch):
    fitting_stocks = []
    try:
        yf_tickers = yf.Tickers(" ".join(batch))
        
        for symbol in batch:
            if symbol is None:
                logging.error("Ticker is empty")
            try:
              info = yf_tickers.tickers[symbol].info
              #print(symbol)
              pe = info.get("trailingPE")
              eps = info.get("trailingEps")

              ps = info.get('priceToSalesTrailing12Months')
              pb = info.get('priceToBook')
              
              #print(pe, eps)
              
              if pe is not None and eps is not None and eps > 0 and pe < eps:
                  pe = round(pe, 2)
                  eps = round(eps, 2)
                  ps = round(ps, 2)
                  pb = round(pb, 2)
                  print(f"MATCH!!!!!!!!!!!!!!!!!!!!!!!!, {symbol}, P/E {pe} < EPS {eps}")
                  logging.critical(f"MATCH!!!!!!!!!!!!!!!!!!!!!!!!, {symbol}, P/E {pe} < EPS {eps}")
                  fitting_stocks.append({"Ticker": symbol,
                                "P/E" : pe,
                                "EPS" : eps,
                                "P/S" : ps,
                                "P/B" : pb
                                        })
              elif eps is None or eps < 0:
                  logging.warning(f"Ticker {symbol} have negative EPS")
                #  print(f"Ticker {symbol} have negative EPS")
              else:
                  logging.info(f"Ticker {symbol} P/E is bigger than EPS, {pe} > {eps}")
                # print(f"Ticker {symbol} EPS is bigger than P/E, {pe} > {eps}")
            except Exception as ex:
              logging.error(ex)
            finally:
                waitPeriod = random.uniform(2.0, 4.0)
                time.sleep(waitPeriod) 
    except Exception as e:
        logging.error(e)
        return fitting_stocks
    return fitting_stocks

In [ ]:
for i in range(0, len(cleaned), batch_size):
    batch = cleaned[i:i + batch_size]
    waitPeriod = random.uniform(2.0, 5.0)
    temp_res = check_the_stock(batch)
    if len(temp_res) > 0:
        results.extend(temp_res)
    if i % 100 == 0:
        print(f"{i} stocks finished")
    time.sleep(waitPeriod)    
results_df = pd.DataFrame(results)
results_df.to_excel(f'../Data/EPS_Bigger_Than_PE_{datetime.now()}.csv', index = False)
#results_df.to_csv(OUTPUT_PATH, index = False, mode = 'a')